<!-- NB_DOC_INTRO_V1 -->
> 📚 **Documentation thematique** : [docs/04_DL.md](docs/04_DL.md) (Dl)
> 📖 **Inventaire global** : [docs/INVENTAIRE.md](docs/INVENTAIRE.md)
> 🗂️ **README projet** : [README.md](README.md)

## A quoi sert ce notebook

Theorie Kolmogorov-Arnold (1957) + implementation naive NumPy + demo `pykan` + comparaison MLP.

---


# Kolmogorov-Arnold Networks (KAN)

> **Référence** : Liu et al., *KAN: Kolmogorov-Arnold Networks*, MIT/Caltech, mai 2024 — https://arxiv.org/abs/2404.19756

## Pourquoi ce notebook ?

Les KAN sont une **alternative théorique aux MLP** apparue en 2024. Idée centrale : remplacer les poids scalaires des arêtes par des **fonctions apprenables** (B-splines), en s'appuyant sur le théorème de Kolmogorov-Arnold.

Promesses :
- 🟢 **Meilleure interprétabilité** (on peut visualiser chaque φ).
- 🟢 **Précision parfois supérieure** sur fonctions analytiques structurées.
- 🟢 **Sparsification** (élagage possible des arêtes ~0).
- 🔴 Lent en pratique (chaque arête = fonction à paramétrer).
- 🔴 Pas (encore) au niveau des MLP/Transformers sur les tâches "réelles" (texte, image).
- 🔴 Implémentations naïves très lentes → utiliser `pykan` (officiel) ou `efficient-kan`.

Plan du notebook :
1. **Théorie** : théorème de Kolmogorov-Arnold (ci-dessous)
2. **Implémentation naïve NumPy** (déjà présente)
3. ➕ Implémentation avec `pykan` (à exécuter)
4. ➕ Comparaison MLP vs KAN sur `make_moons`
5. ➕ Limites et alternatives


Les Kolmogorov-Arnold Networks (KAN) sont un cadre théorique pour l'approximation de fonctions continues sur un intervalle donné. Elles se basent sur les travaux de Kolmogorov et Arnold, qui ont démontré des résultats fondamentaux sur la représentation des fonctions continues.

### Théorème de Kolmogorov

Le théorème de Kolmogorov énonce que toute fonction continue multivariée peut être représentée comme une superposition de fonctions continues univariées. Plus formellement, il affirme que pour toute fonction continue $ f: [0,1]^n→R $, il existe des fonctions continues univariées $ϕ_i$  et $ψ_{ij}$ telles que :

$f(x_1,x_2,...,x_n)=\sum_{i=1}^{2n+1}​ϕ_i (\sum_{j=1}^{n} ​ψ_{ij}(x_j))$

### Contribution de Arnold

Arnold a amélioré ce résultat en montrant que les fonctions $ψ_{ij}$​ peuvent être choisies de manière plus structurée et avec moins de dépendances.

### Application aux réseaux de neurones

Les Kolmogorov-Arnold Networks (KAN) sont des réseaux de neurones qui exploitent ce théorème pour construire des modèles capables d'approximer des fonctions continues. La structure d'un KAN est généralement composée de trois couches :

1.  **Entrée** : La couche d'entrée prend les variables $f(x_1,x_2,...,x_n)$​.
2.  **Transformation intermédiaire** : Chaque entrée est transformée par un ensemble de fonctions univariées $​ψ_{ij} $​.
3.  **Superposition** : Les sorties des transformations intermédiaires sont combinées par des fonctions univariées $ϕ_i$​ pour produire la sortie finale.

### Avantages des KAN

*   **Théoriquement fondé** : Les KAN reposent sur un théorème mathématique robuste, garantissant une approximation universelle pour des fonctions continues.
*   **Simplicité des transformations univariées** : La décomposition en transformations univariées simplifie l'analyse et la formation du réseau.

### Limites des KAN

*   **Complexité de construction** : Trouver les fonctions $ϕ_i$ ​ et $ψ_{ij}$​ adaptées peut être complexe et non trivial dans la pratique.
*   **Scalabilité** : Pour des problèmes de grande dimension, la mise en œuvre des KAN peut devenir impraticable en raison de la croissance rapide du nombre de fonctions nécessaires.

In [1]:
import numpy as np

def psi(x, j):
    """Fonction de transformation univariée ψ_ij"""
    return np.sin(x + j)

def phi(y, i):
    """Fonction de superposition univariée φ_i"""
    return np.sum(y) + i

def kolmogorov_arnold_network(X):
    """Réseau Kolmogorov-Arnold approchant une fonction continue"""
    n = X.shape[1]
    output = 0
    for i in range(2 * n + 1):
        sum_psi = np.zeros(X.shape[0])
        for j in range(n):
            sum_psi += psi(X[:, j], j)
        output += phi(sum_psi, i)
    return output

# Exemple d'utilisation
X = np.array([[0.1, 0.2], [0.4, 0.5], [0.7, 0.8]])
result = kolmogorov_arnold_network(X)
print(result)


30.184255748213225


---

## 3. Implémentation pratique avec `pykan`

L'implémentation naïve ci-dessus utilise `ψ(x,j) = sin(x+j)` et `φ(y,i) = sum(y)+i` — c'est purement didactique, pas apprenable. **`pykan`** (la lib officielle du papier original) apprend des **B-splines** sur chaque arête.

> Installation : `pip install pykan` (récupère torch en dépendance).


In [ ]:
# Demo pykan — necessite pip install pykan
# Decommenter pour executer

# !pip install -q pykan torch

# from kan import KAN
# import torch

# # Petit dataset : approxime f(x1, x2) = sin(pi*x1) * cos(pi*x2)
# torch.manual_seed(0)
# n = 1000
# X = torch.rand((n, 2)) * 2 - 1  # [-1, 1]
# y = torch.sin(torch.pi * X[:, 0]) * torch.cos(torch.pi * X[:, 1])
# y = y.unsqueeze(1)

# # KAN : 2 inputs, 5 nodes hidden, 1 output
# # grid=5 = nombre de noeuds des splines
# # k=3 = ordre des splines (cubiques)
# model = KAN(width=[2, 5, 1], grid=5, k=3, seed=0)

# # Train (sans dataset loader, direct)
# def train(model, X, y, steps=200, lr=0.01):
#     optimizer = torch.optim.LBFGS(model.parameters(), lr=lr)
#     for step in range(steps):
#         def closure():
#             optimizer.zero_grad()
#             loss = ((model(X) - y) ** 2).mean()
#             loss.backward()
#             return loss
#         loss = optimizer.step(closure)
#         if step % 20 == 0:
#             print(f"step {step:4d} | mse {loss.item():.5f}")
#     return model

# model = train(model, X, y)

# # Visualisation des splines apprises
# model.plot(scale=2)  # ouvre un plot interactif des phi/psi par arete

# # Sparsification (elaguer les aretes ~0)
# model.prune()
# model.plot(scale=2)


---

## 4. Comparaison MLP vs KAN — démarche

Sur des **fonctions analytiques structurées** (sin, cos, polynômes, leurs combinaisons), KAN bat souvent MLP à paramètres équivalents — c'est le résultat phare du papier.

Sur des **données réelles bruitées** (images, texte), MLP reste mieux ou équivalent.

### Mini-bench MLP vs KAN sur sklearn `make_moons`


In [ ]:
# Decommenter pour executer (necessite pykan + torch + sklearn)

# from sklearn.datasets import make_moons
# from sklearn.model_selection import train_test_split
# import torch, torch.nn as nn

# X, y = make_moons(n_samples=2000, noise=0.2, random_state=0)
# X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=0)
# X_tr_t, X_te_t = torch.tensor(X_tr, dtype=torch.float32), torch.tensor(X_te, dtype=torch.float32)
# y_tr_t, y_te_t = torch.tensor(y_tr, dtype=torch.float32).unsqueeze(1), torch.tensor(y_te, dtype=torch.float32).unsqueeze(1)

# # --- MLP baseline ---
# class MLP(nn.Module):
#     def __init__(self):
#         super().__init__()
#         self.net = nn.Sequential(
#             nn.Linear(2, 8), nn.ReLU(),
#             nn.Linear(8, 1), nn.Sigmoid()
#         )
#     def forward(self, x): return self.net(x)

# mlp = MLP()
# opt = torch.optim.Adam(mlp.parameters(), lr=1e-2)
# for ep in range(200):
#     pred = mlp(X_tr_t)
#     loss = nn.functional.binary_cross_entropy(pred, y_tr_t)
#     opt.zero_grad(); loss.backward(); opt.step()

# mlp_acc = ((mlp(X_te_t) > 0.5).float() == y_te_t).float().mean().item()
# print(f"MLP test acc : {mlp_acc:.3f} ({sum(p.numel() for p in mlp.parameters())} params)")

# # --- KAN ---
# # from kan import KAN
# # kan = KAN(width=[2, 4, 1], grid=5, k=3, seed=0)
# # ... (train via LBFGS comme plus haut, avec BCE)


---

## 5. Alternatives modernes

| Implémentation | Backbone | Vitesse | Maturité |
|---|---|---|---|
| **`pykan`** (officiel MIT) | PyTorch | Modérée | Référence du papier |
| **`efficient-kan`** (Blealtan) | PyTorch | Rapide (10-20× pykan) | Bibliothèque communautaire populaire |
| **`fast-kan`** | PyTorch + JIT | Très rapide | Optimisée pour gros modèles |
| **`fourier-kan`** | PyTorch | Fast | Remplace splines par Fourier features |

## 6. Limites et perspectives

### Limites observées (juin 2024 — 2025)
- **Coût mémoire** : explose avec largeur × grid × ordre des splines.
- **Bench grands modèles** : MLP/Transformer restent supérieurs sur tâches NLP/vision.
- **Pas de boost transfert** : pas d'analogue à BERT/CLIP pré-entraînés pour KAN.

### Bons cas d'usage potentiels
- **PINNs** (Physics-Informed Neural Networks) — fonctions analytiques connues.
- **Découverte symbolique** — extraire la formule à partir des splines apprises (`model.symbolic_formula()`).
- **Tabulaire structuré** avec interactions complexes connues.

### Conseil
Si tu débutes : **MLP + XGBoost** restent les baselines à battre. KAN = curiosité de R&D fin 2024 / 2025.

## Voir aussi

- Liu et al., 2024. *KAN: Kolmogorov-Arnold Networks*. https://arxiv.org/abs/2404.19756
- KAN 2.0 (suite, août 2024) : extensions et accélérations. https://arxiv.org/abs/2408.10205
- Repo officiel : https://github.com/KindXiaoming/pykan
